In [ ]:
import os
from pathlib import Path

import pandas as pd
pd.set_option('display.max_colwidth', None)  # Allows unlimited column width
pd.set_option('display.width', 2000)  # Increases the total display width
pd.set_option('display.max_rows', None)

# Change directory
# Modify this cell to insure that the output shows the correct path.
# Define all paths relative to the project root shown in the cell output
# project_root = "/Users/jackienguyen/Desktop/DLVR/freqtrade"
project_root = "/workspaces/freqtrade"
try:
    os.chdir(project_root)
    if not Path("LICENSE").is_file():
        i = 0
        while i < 4 and (not Path("LICENSE").is_file()):
            os.chdir(Path(Path.cwd(), "../"))
            i += 1
        project_root = Path.cwd()
except FileNotFoundError:
    print("Please define the project root relative to the current directory")
print(Path.cwd())

In [ ]:
# accept current iteration with not perfect, but we will go ahead

# next time: strat.ninja get dca strat as well, current we only get the public strat
# we also ignore the github_scraped_strategies. It helps us get a feel of the market, but strat.ninja might already include everything.

# the repo provide utils, should use it

# trial on 15m and 1h as well

# pick out 4 best strat

# dry run

In [ ]:
# ↓ this list literal is written in ascending-price order, so no sort() later
price_levels = [
    ("p100", 0.0,   0),
    ("p95",  0.16,  0),
    ("p90",  0.314, 0),
    ("p85",  0.473, 0),
    ("p80",  0.627, 1),
    ("p75",  0.793, 2),
    ("p70",  0.949, 4),
    ("p60",  1.251, 6),
    ("p50",  1.575, 8),
    ("p40",  1.882, 10),
    ("p30",  2.206, 10),
    ("p20",  2.530, 10),
    ("p10",  2.837, 10),
    ("p0",   3.158, 10),
]

In [1]:
price_levels_settings = {
    "_p0": {"price": 3.158, "max_nb_order": 10},
    "_p10": {"price": 2.837, "max_nb_order": 10},
    "_p20": {"price": 2.530, "max_nb_order": 10},
    "_p30": {"price": 2.206, "max_nb_order": 10},
    "_p40": {"price": 1.882, "max_nb_order": 10},
    "_p50": {"price": 1.575, "max_nb_order": 8},
    "_p60": {"price": 1.251, "max_nb_order": 6},
    "_p70": {"price": 0.949, "max_nb_order": 4},
    "_p75": {"price": 0.793, "max_nb_order": 2},
    "_p80": {"price": 0.627, "max_nb_order": 1}, # DCA threshold
    "_p85": {"price": 0.473, "max_nb_order": 0},
    "_p90": {"price": 0.314, "max_nb_order": 0},
    "_p95": {"price": 0.16, "max_nb_order": 0},
    "_p100": {"price": 0.0, "max_nb_order": 0},
}
price_levels_max_nb_orders = price_levels_settings

In [3]:

def get_price_levels_max_nr_orders(current_rate: float, price_levels_max_nb_orders: dict) -> (str, int):
    levels=sorted(price_levels_max_nb_orders.items(),key=lambda x:x[1]["price"])
    for i in range(len(levels)-1):
        lower_k,lower_v=levels[i]; upper_k,upper_v=levels[i+1]
        if lower_v["price"]<=current_rate<upper_v["price"]:
            return f"{lower_k[1:]}_{upper_k[1:]}",upper_v["max_nb_order"]

    top_k, top_v=levels[-1]
    if current_rate>=top_v["price"]: return top_k[1:], top_v["max_nb_order"]
    return None

get_price_levels_max_nr_orders(current_rate=0.7, price_levels_max_nb_orders=price_levels_max_nb_orders)


('p80_p75', 2)

In [20]:
import pandas as pd, numpy as np

df = pd.DataFrame({
    'date': pd.date_range('2025-05-01', periods=15),
    'enter_long': [1.0, np.nan, np.nan, np.nan, np.nan, 1.0, 1.0, 1.0, np.nan, np.nan, 1.0, 1.0, 1.0, np.nan, 1.0]
})

print(df)

# arr = df['enter_long'].to_numpy()
# nan = np.isnan(arr)
# runs = nan & np.concatenate(([False], nan[:-1]))
# idx = np.where(runs)[0]
# print("Last NaN in a run at index", idx[-1], "with date", df['date'].iloc[idx[-1]])

         date  enter_long
0  2025-05-01         1.0
1  2025-05-02         NaN
2  2025-05-03         NaN
3  2025-05-04         NaN
4  2025-05-05         NaN
5  2025-05-06         1.0
6  2025-05-07         1.0
7  2025-05-08         1.0
8  2025-05-09         NaN
9  2025-05-10         NaN
10 2025-05-11         1.0
11 2025-05-12         1.0
12 2025-05-13         1.0
13 2025-05-14         NaN
14 2025-05-15         1.0


In [21]:
ser = df['enter_long']
# only proceed if last row is -1.0
if ser.iat[-1] == 1.0:
    # mask where this row AND the previous are NaN
    print("xxx")
    mask = ser.isna() & ser.shift(periods=1).isna() & ser.shift(periods=2).isna()
    if mask.any():
        last_idx = mask[mask].last_valid_index()
        print("Found at index label", last_idx)
    else:
        print("No back-to-back NaNs found")

xxx
Found at index label 4


In [23]:
df.iloc[-11]

date          2025-05-05 00:00:00
enter_long                    NaN
Name: 4, dtype: object

In [22]:
4 - len(df)

-11

In [ ]:
# import os

# def find_file(strategy_folder):
#     """
#     Recursively searches for a file in the given directory and returns its full path.

#     Args:
#         filename (str): The name of the file to search for.
#         directory (str): The directory to start the search from.

#     Returns:
#         str or None: The full path to the file if found, otherwise None.
#     """
#     # for root, _, files in os.walk(directory):
#     #     if filename in files:
#     #         return os.path.join(root, filename)
#     # return None
#     strategy_names = []
#     # pattern = re.compile(r'class\s+(\w+)\s*\(\s*IStrategy\s*\)')
#     pattern = re.compile(r'FreqCtrlBTC')
#     for root, _, files in os.walk(strategy_folder):
#         for file in files:
#             if file.endswith(".py"):
#                 file_path = os.path.join(root, file)
#                 with open(file_path, "r", encoding="utf-8") as f:
#                     content = f.read()
#                     matches = pattern.findall(content)
#                     if matches:
#                         strategy_names.append(file_path)
#                     else: pass
#     return strategy_names
#                     # strategy_names.extend(matches)
#                     # if filename in matches:
#                     #     return file_path



# # Example usage:
# directory_to_search = Path(project_root) / "user_data/strategies/github_scraped_strategies/Sshimaninja_Freqtrade_Public"  # Folder containing strategies

# file_path = find_file(directory_to_search)
# if file_path:
#     print(f"File found: {file_path}")
# else:
#     print("File not found.")